# Four-segment business decision and audit workbook

Run this notebook in Google Colab with **Runtime > Run all**. It loads a pinned version of business_analysis.py and the already fitted k=4 model. It does not fit new centers or change the training/test split.

The professor asks for Python code that reproduces every number. This notebook and the Python module provide that code. They export an Excel workbook with formulas and CSV tables for checking the analysis. The final submission remains a title page plus two student-written content slides in one PDF.

The working names can be revised in the Python module. Eligibility is chosen from training members only: credit a segment if its training contribution is positive; a zero contribution is a no-credit tie. Values for all 330 members are then calculated under that fixed policy.

In [ ]:
%pip install -q numpy pandas xlsxwriter openpyxl

## 1. Load the reproducible Python code

The code revision is fixed below. The module records a SHA-256 digest of its source and the input-data/model revisions in selected_model.json.

In [ ]:
import hashlib
import importlib.util
import json
import shutil
from pathlib import Path
from urllib.request import urlopen
from IPython.display import display

SOURCE_REF = "b1951a9d085658befce8566a49b1fb958be2253a"
SOURCE_URL = f"https://raw.githubusercontent.com/FrankZhao111/ClusterHW/{SOURCE_REF}/business_analysis.py"
with urlopen(SOURCE_URL) as response:
    source_bytes = response.read()
module_file = Path("business_analysis.py")
module_file.write_bytes(source_bytes)
spec = importlib.util.spec_from_file_location("gym_business_analysis", module_file)
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
print('Python code revision:', SOURCE_REF)
print('Source SHA-256:', hashlib.sha256(source_bytes).hexdigest())

## 2. Run the decision and create the Excel/CSV outputs

Finance's rule is applied member by member: below 20 dollars of existing extras spend, incremental contribution is 25 x 60% - 10 = +5 dollars; at or above 20, contribution is -10 dollars.

The baseline credits everyone. V1 credits only segments with positive training-member contribution. The new member is a separate example and is excluded from the 330-member financial totals.

The code checks nearest-center assignments, confirms the one-member function agrees for all 330 members, reconciles financial totals, and reads back the exported workbook's formula caches and formula cells. Excel recalculates the actual workbook formulas when the workbook is opened.

In [ ]:
results = analysis.run_analysis()
print('Model fit: baseline versus selected k=4')
display(results['tables']['model_fit_summary'].round(4))
print('Segment descriptions: counts cover all 330 members; traits are training centroids')
display(results['tables']['segment_summary'].round(4))
print('Training-based credit decisions and full-population economics')
display(results['tables']['segment_policy'].round(4))
print('Monthly policy comparison under the supplied assumptions')
display(results['tables']['plan_comparison'].round(2))
print('Validation:', results['validation'])
print('Workbook:', results['workbook_path'])

## 3. Predict the supplied new member

make_recommender returns a function named recommend_offer with one argument: the member's six features. The function applies the training scaler, finds the nearest saved center, and returns the segment and its credit eligibility.

You can edit the example dictionary in this cell to inspect another member. This does not add that example to the historical members or alter the policy.

In [ ]:
recommend_offer = analysis.make_recommender(results['selected_model'])
example_member = {
    'age': 30,
    'visits_per_month': 4,
    'avg_visit_minutes': 42,
    'classes_per_month': 1,
    'months_member': 5,
    'monthly_extras_spend': 10,
}
print(recommend_offer(example_member))
print('Scaling calculations for the assignment example:')
display(results['tables']['new_member_scaling'].round(6))
print('Distances from the assignment example to each of the four centers:')
display(results['tables']['new_member_distances'].round(6))

## 4. Read the workbook

Start with Guide, Model_Fit, Segment_Summary, Segment_Policy, Plan_Comparison, and New_Member. Other sheets provide the detailed records and formula audit.

- Segment_Summary describes the training centroids; Full_Profiles shows means across all assigned members.
- Members contains one row per historical member, including both policies' contribution calculations.
- Segment_Policy shows why each segment qualifies or does not qualify using training members.
- Plan_Comparison reconciles incremental revenue, margin, credit cost, net contribution, and improvement.
- New_Member shows the six raw inputs, standardized values, distances, selected segment, and credit decision.
- Fit_Comparison includes the eight candidate spreads and editable native Excel elbow charts.

The Excel formulas contain cached results supplied by Python for preview applications. Changing blue assumption or new-member input cells in Excel is a what-if calculation; the exported CSVs retain this notebook run's values.

Expected reference results for the supplied data and assumptions: baseline net -1200 dollars/month, V1 net +185 dollars/month, improvement +1385 dollars/month. Credit goes to cluster 0, and the supplied new member is assigned to cluster 0 and receives the credit. These are simulation results under the assignment's assumptions, not measured campaign outcomes.

In [ ]:
output_dir = results['output_dir']
shutil.copy2(module_file, output_dir / 'business_analysis.py')
(output_dir / 'validation_checks.json').write_text(
    json.dumps(results['validation'], indent=2) + '\n', encoding='utf-8'
)
archive_path = shutil.make_archive('business_outputs', 'zip', root_dir=output_dir)
print('Files in the ZIP:')
for path in sorted(output_dir.iterdir()):
    print(' -', path.name)
print('Archive:', archive_path)
from google.colab import files
files.download(archive_path)

## Preserve the run

Download and extract business_outputs.zip, then upload its Excel workbook, CSV tables, JSON files, and Python source to GitHub in a business_outputs folder. Save the executed notebook to GitHub, or download the executed .ipynb and upload it.

Use these verified numerical outputs to build your own two content slides. Review the assumptions, centroids versus full-population averages, and member-level contribution calculations before choosing which figures to present.